In [1]:
import torch
from torch import nn

net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.rand(size=(2, 4))
net(X)

tensor([[0.0565],
        [0.0659]], grad_fn=<AddmmBackward0>)

### 1. 参数访问

In [2]:
print(net[2].state_dict())
net[0].weight.shape


OrderedDict([('weight', tensor([[ 0.3069, -0.0167, -0.1531, -0.1876, -0.1387, -0.2050,  0.2498,  0.2073]])), ('bias', tensor([0.2511]))])


torch.Size([8, 4])

nn.Linear 内部实际存储的是 W^T，shape 是 (8,4)

In [3]:
net[2].state_dict()

OrderedDict([('weight',
              tensor([[ 0.3069, -0.0167, -0.1531, -0.1876, -0.1387, -0.2050,  0.2498,  0.2073]])),
             ('bias', tensor([0.2511]))])

In [4]:
print(net.named_parameters())

<generator object Module.named_parameters at 0x00000189B9ED9310>


In [5]:
net[2].bias

Parameter containing:
tensor([0.2511], requires_grad=True)

In [6]:
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.2511], requires_grad=True)
tensor([0.2511])


In [7]:
net[2].weight.grad == None

True

In [8]:
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [9]:
net.state_dict()['2.bias']

tensor([0.2511])

In [10]:
net.parameters, net.state_dict()

(<bound method Module.parameters of Sequential(
   (0): Linear(in_features=4, out_features=8, bias=True)
   (1): ReLU()
   (2): Linear(in_features=8, out_features=1, bias=True)
 )>,
 OrderedDict([('0.weight',
               tensor([[-0.1880,  0.1477, -0.4307,  0.2118],
                       [ 0.3973, -0.2373,  0.1140,  0.1052],
                       [ 0.4087,  0.0884, -0.0029,  0.0123],
                       [-0.0913, -0.1288,  0.0392, -0.3493],
                       [ 0.1565,  0.0623,  0.4831,  0.4132],
                       [ 0.3202,  0.0933, -0.4012,  0.2393],
                       [ 0.3531, -0.4918, -0.0268, -0.1069],
                       [ 0.4964, -0.4076, -0.3689,  0.2927]])),
              ('0.bias',
               tensor([-0.4264, -0.1796,  0.4906,  0.0124,  0.2587, -0.4485, -0.0875,  0.0080])),
              ('2.weight',
               tensor([[ 0.3069, -0.0167, -0.1531, -0.1876, -0.1387, -0.2050,  0.2498,  0.2073]])),
              ('2.bias', tensor([0.2511]))]))

In [11]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在这里嵌套
        net. add_module(f'block{i}', block1())
    return net

rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
rgnet(X)
#print(rgnet.modules)
#print(rgnet)

tensor([[0.2249],
        [0.2252]], grad_fn=<AddmmBackward0>)

In [12]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [13]:
rgnet[0][1][0].bias.data

tensor([-0.3174, -0.3251,  0.3841,  0.1310,  0.2877, -0.0142,  0.1769, -0.4774])

### 2. 参数初始化

In [15]:
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([-0.0014,  0.0054, -0.0078,  0.0173]), tensor(0.))

In [17]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)
net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [19]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant(m.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)

tensor([ 0.1800,  0.2500,  0.2255, -0.2476])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


C:\Users\xzsss\AppData\Local\Temp\ipykernel_19708\2923948313.py:6: FutureWarning: `nn.init.constant` is now deprecated in favor of `nn.init.constant_`.
  nn.init.constant(m.weight, 42)


In [21]:
def my_init(m):
    if type(m) == nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight, -10, 10)
        m.weight.data *= m.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[-0.0000,  0.0000,  0.0000,  0.0000],
        [ 5.2227,  0.0000, -8.2808,  0.0000]], grad_fn=<SliceBackward0>)

In [22]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.,  1.,  1.,  1.])

### 3. 参数绑定

In [27]:
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0]

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


tensor([ 1.0000e+02, -2.8708e-01, -6.1731e-02,  1.5857e-01, -3.2266e-01,
        -3.3729e-01,  1.9017e-01, -2.1232e-01])

当参数绑定时，梯度会发生什么情况？ 答案是由于模型参数包含梯度，因此在反向传播期间第二个隐藏层 （即第三个神经网络层）和第三个隐藏层（即第五个神经网络层）的梯度会加在一起。